<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/midterm/MID3_goodhart_side_effects.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Ethics, Fairness and Specification in Practice

## Project 3: Goodhart's law, side effects and impact penalties in reinforcement learning

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A Q-learning agent in a small gridworld can reach its goal quickly by breaking a vase that the proxy reward ignores [1]. The example varies an impact penalty and reports proxy and true returns, and it simulates regressional Goodhart by selecting among options with a noisy proxy under increasing optimisation pressure [2].

**Required extensions.** Replace the hand-coded penalty with a general impact measure that does not name the vase, for example one based on the reachability of states or on attainable utility for auxiliary goals [3], and test it in at least two new environments with different side effects. Add a second Goodhart variant, analyse the trade-off between task performance and caution, and relate the findings to the formal definition of reward hacking [4].

## What this example implements

The notebook trains tabular Q-learning agents for a range of impact penalties, compares proxy and true returns and plots how the true value of the option selected by a noisy proxy falls behind its proxy score.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Side effects in a gridworld

In [ ]:
GRID = ["S.V.G",
        ".###.",
        "....."]
H, W = len(GRID), len(GRID[0])
VASE, GOAL = (0, 2), (0, 4)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step(state, a):
    r, c, intact = state
    nr, nc = r + MOVES[a][0], c + MOVES[a][1]
    if not (0 <= nr < H and 0 <= nc < W) or GRID[nr][nc] == "#":
        nr, nc = r, c
    broke = intact and (nr, nc) == VASE
    return (nr, nc, intact and not broke), broke, (nr, nc) == GOAL

def q_learning(lam=0.0, episodes=3000, alpha=0.5, gamma=0.99, seed=SEED):
    rng = np.random.default_rng(seed)
    Q = np.zeros((H, W, 2, 4))
    for ep in range(episodes):
        s, eps = (0, 0, True), max(0.05, 1 - ep / 1500)
        for _ in range(50):
            q = Q[s[0], s[1], int(s[2])]
            a = int(rng.integers(4)) if rng.random() < eps else int(np.argmax(q))
            s2, broke, done = step(s, a)
            r = (1.0 if done else 0.0) - 0.01 - lam * broke
            target = r + (0.0 if done else gamma * Q[s2[0], s2[1], int(s2[2])].max())
            q[a] += alpha * (target - q[a])
            s = s2
            if done:
                break
    return Q

def rollout(Q):
    s, proxy, true, path = (0, 0, True), 0.0, 0.0, [(0, 0)]
    for _ in range(30):
        s, broke, done = step(s, int(np.argmax(Q[s[0], s[1], int(s[2])])))
        proxy += (1.0 if done else 0.0) - 0.01
        true += (1.0 if done else 0.0) - 0.01 - 1.0 * broke
        path.append(s[:2])
        if done:
            break
    return proxy, true, path

for lam in [0.0, 0.02, 0.05, 0.1, 0.2, 0.5]:
    proxy, true, path = rollout(q_learning(lam))
    print(f"impact penalty {lam:>4}: proxy return {proxy:.2f}, true return {true:.2f}, path length {len(path) - 1}")

## 2. Regressional Goodhart under selection

In [ ]:
rng = np.random.default_rng(SEED)
Ns = [1, 2, 5, 10, 20, 50, 100, 200]
res = []
for N in Ns:
    tv, pv = [], []
    for _ in range(2000):
        v = rng.normal(0, 1, N)
        proxy = v + rng.normal(0, 1, N)
        i = int(np.argmax(proxy))
        tv.append(v[i]); pv.append(proxy[i])
    res.append((N, np.mean(pv), np.mean(tv)))
res = np.array(res)
plt.semilogx(res[:, 0], res[:, 1], "o-", label="proxy score of the selected option")
plt.semilogx(res[:, 0], res[:, 2], "s-", label="true value of the selected option")
plt.xlabel("number of options compared (optimisation pressure)"); plt.ylabel("mean value"); plt.legend(); plt.title("Regressional Goodhart"); plt.show()

## Report and submission

The technical report states the question, explains the design choices, presents the experiments with the figures produced by the notebook and discusses the ethical and safety implications and the limitations of the results. It cites at least eight sources in square brackets, at least four of them from the course readings, and links to the final Colab notebook.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Leike, J., Martic, M., Krakovna, V., Ortega, P. A., Everitt, T., Lefrancq, A., Orseau, L., & Legg, S. (2017). AI safety gridworlds. arXiv preprint arXiv:1711.09883. <https://arxiv.org/abs/1711.09883>

[2] Manheim, D., & Garrabrant, S. (2018). Categorizing variants of Goodhart's law. arXiv preprint arXiv:1803.04585. <https://arxiv.org/abs/1803.04585>

[3] Turner, A. M., Hadfield-Menell, D., & Tadepalli, P. (2020). Conservative agency via attainable utility preservation. In *Proceedings of the AAAI/ACM Conference on AI, Ethics, and Society (AIES '20)* (pp. 385-391). ACM.

[4] Skalse, J., Howe, N. H. R., Krasheninnikov, D., & Krueger, D. (2022). Defining and characterizing reward hacking. In *Advances in Neural Information Processing Systems 35*.